In [30]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
import json
from dotenv import load_dotenv
import os

load_dotenv()

True

In [31]:
llm = init_chat_model(
    model=os.getenv('MOONSHOT_MODEL_ID'), 
    model_provider='openai',
    api_key=os.getenv("MOONSHOT_API_KEY"),
    base_url=os.getenv("MOONSHOT_BASE_URL"),
    temperature=0.9
)

In [23]:
class MultiQueryComposer(BaseModel):
    """用于将用户的查询分解为一个或多个子查询，方便后续检索"""
    analyze: str = Field(description="你对于用户查询的拆分思考过程")
    queries: list[str] = Field(description="每个元素分别是分解后的子查询")

In [27]:
query = "今天世界杯，我晚上想熬夜看球，应该吃点啥？我应该准备点啥食材呢？"

multi_query_composer_prompt = f"""
你是一个美食系统的智能助手，美食系统负责回答用户关于菜品推荐、菜谱生成、菜品问答三方面的提问。

要求：
 - 用户的提问中可能包含多个问题，你需要对用户的问题进行分析并在必要的时候将其拆分成多个子问题
 - 你需要将拆分出来的子问题填入JSON对象中的 'queries' 字段，该字段类型为字符串列表

如：
 - 用户提问：我想吃点清淡的菜，有什么推荐吗？还有宫保鸡丁怎么做？
   你需要拆分成两个子问题：[推荐清淡菜, 宫保鸡丁的做法]
 - 用户提问：宫保鸡丁和红烧茄子哪个好吃？
   你需要拆分成两个子问题：[宫保鸡丁怎么样, 红烧茄子怎么样]"""

messages = [
    {'role': 'system', 'content': multi_query_composer_prompt}, 
    {'role': 'user', 'content': query}
]

In [33]:
response = llm.invoke(
    input=messages,
    temperature=0.3, 
    response_format=MultiQueryComposer, 
)

In [35]:
json.loads(response.content)

{'analyze': '用户的问题包含两个部分：1. 熬夜看球时适合吃什么；2. 需要提前准备哪些食材。因此拆成两个子问题。',
 'queries': ['熬夜看球吃什么', '熬夜看球需要准备哪些食材']}